# Rally E2B RP Merged Upload

Recreates the Rally E2B A100/B75 merged checkpoint from the two-stage SFT Kaggle output, validates the checkpoint files, and uploads it to Hugging Face when `HF_TOKEN` is available as a Kaggle secret. If the HF secret is unavailable, the notebook still writes a provenance report and keeps the merged checkpoint in Kaggle output.

In [ ]:
import os, platform, shutil
from pathlib import Path

print('python_platform=', platform.platform())
print('working_disk_free_gb=', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2))
print('input_dirs=', [str(p) for p in Path('/kaggle/input').glob('*')])


In [ ]:
import os, subprocess, sys, time

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    '--upgrade',
    'huggingface_hub[cli]>=1.5.0',
    'hf_transfer>=0.1.9',
    'safetensors>=0.4.5',
])

secret_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN') or ''
secret_attempts = int(os.environ.get('HF_SECRET_ATTEMPTS', '3'))
for attempt in range(secret_attempts):
    if secret_token:
        break
    try:
        from kaggle_secrets import UserSecretsClient
        secret_token = UserSecretsClient().get_secret('HF_TOKEN')
        break
    except Exception as exc:
        print('hf_secret_attempt_failed=', attempt + 1, type(exc).__name__)
        time.sleep(10)
if secret_token and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = secret_token
if secret_token and not os.environ.get('HUGGING_FACE_HUB_TOKEN'):
    os.environ['HUGGING_FACE_HUB_TOKEN'] = secret_token
print('hf_secret_loaded=', bool(secret_token))
print('hf_upload_enabled=', bool(os.environ.get('HF_TOKEN')))


In [ ]:
import json, os, subprocess, sys
from pathlib import Path

REPO_URL = os.environ.get('HERETIC_TO_ONNX_REPO', 'https://github.com/alkahest-ai/heretic-to-onnx.git')
REPO_REF = os.environ.get('HERETIC_TO_ONNX_REF', 'codex/kaggle-heretic-2b-run')
REPO_DIR = Path('/kaggle/working/heretic-to-onnx')
if REPO_DIR.exists():
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])
else:
    subprocess.check_call(['git', 'clone', '--branch', REPO_REF, '--depth', '1', REPO_URL, str(REPO_DIR)])
print('repo_head=', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())

artifact_name = os.environ.get('RALLY_TWO_STAGE_ARTIFACT_NAME', 'rally-e2b-two-stage-sft')
explicit_artifact_dir = os.environ.get('RALLY_ARTIFACT_DIR', '').strip()
candidates = [
    Path(explicit_artifact_dir) if explicit_artifact_dir else None,
    Path('/kaggle/input') / artifact_name,
    Path('/kaggle/input') / artifact_name / artifact_name,
    Path('/kaggle/input/notebooks') / artifact_name,
    Path('/kaggle/input/notebooks') / artifact_name / artifact_name,
]
candidates.extend(Path('/kaggle/input').glob(f'**/{artifact_name}'))
candidates = [path for path in candidates if path is not None]

def has_artifacts(path):
    return (
        (path / 'stage-a-adapter' / 'adapter_model.safetensors').exists()
        and (path / 'stage-b-adapter' / 'adapter_model.safetensors').exists()
        and ((path / 'stage-a-merged' / 'model.safetensors').exists() or (path / 'stage-a-merged' / 'model.safetensors.index.json').exists())
    )

artifact_dir = next((path for path in candidates if has_artifacts(path)), None)
if artifact_dir is None:
    checked = '\n'.join(str(path) for path in candidates)
    raise FileNotFoundError(f'Could not find completed Rally E2B two-stage artifacts. Checked:\n{checked}')

merged_dir = Path(os.environ.get('RALLY_MERGED_OUTPUT_DIR', '/kaggle/working/a100-b75-merged'))
stage_b_scale = float(os.environ.get('RALLY_STAGE_B_SCALE', '0.75'))
if merged_dir.exists():
    import shutil
    shutil.rmtree(merged_dir, ignore_errors=True)
subprocess.check_call([
    sys.executable,
    str(REPO_DIR / 'scripts/merge_lora_scaled.py'),
    '--base-dir', str(artifact_dir / 'stage-a-merged'),
    '--adapter-dir', str(artifact_dir / 'stage-b-adapter'),
    '--output-dir', str(merged_dir),
    '--scale', str(stage_b_scale),
], cwd=str(REPO_DIR))

files = [
    {
        'name': path.name,
        'size': path.stat().st_size,
    }
    for path in sorted(merged_dir.iterdir(), key=lambda item: item.name)
    if path.is_file()
]
file_names = {item['name'] for item in files}
checkpoint_ok = bool(
    'model.safetensors' in file_names
    or 'model.safetensors.index.json' in file_names
    or any(name.endswith('.safetensors') for name in file_names)
)
required_ok = checkpoint_ok and 'config.json' in file_names and 'tokenizer_config.json' in file_names
repo_id = os.environ.get('RALLY_RP_MERGED_REPO', 'thomasjvu/rally-2b-rp-a100-b75-merged')
private = os.environ.get('RALLY_PRIVATE', '1') != '0'
upload = {'ok': False, 'skipped': True, 'reason': 'HF_TOKEN is not set'}
if os.environ.get('HF_TOKEN'):
    from huggingface_hub import HfApi
    api = HfApi(token=os.environ['HF_TOKEN'])
    api.create_repo(repo_id=repo_id, repo_type='model', private=private, exist_ok=True)
    commit = api.upload_folder(
        repo_id=repo_id,
        repo_type='model',
        folder_path=str(merged_dir),
        commit_message='Upload Rally E2B two-stage RP A100/B75 merged checkpoint from Kaggle',
    )
    upload = {'ok': True, 'skipped': False, 'commit_url': str(commit)}
report = {
    'ok': required_ok and (upload['ok'] or upload['skipped']),
    'repo_id': repo_id,
    'private': private,
    'artifact_dir': str(artifact_dir),
    'stage_b_scale': stage_b_scale,
    'source_dir': str(merged_dir),
    'checkpoint_ok': checkpoint_ok,
    'required_ok': required_ok,
    'upload': upload,
    'files': files,
}
report_path = Path('/kaggle/working/rally-e2b-rp-merged-upload-report.json')
report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + '\n')
print(json.dumps(report, indent=2, sort_keys=True))
